Is the apparent stacking improvement stable, and how exactly are semantic and structured models complementary?

In [3]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/taxonomy_train.csv")
test_df = pd.read_csv("../data/processed/taxonomy_test.csv")

# ---------------------------------------------------------
# 1. Expected sizes
# ---------------------------------------------------------

assert len(train_df) == 1489
assert len(test_df) == 287
assert len(train_df) + len(test_df) == 1776

# ---------------------------------------------------------
# 2. Expected labels
# ---------------------------------------------------------

expected_train = {
    "workflow_error": 660,
    "constraint_error": 317,
    "tool_use_error": 237,
    "grounding_state_error": 244,
    "reasoning_value_error": 31,
}

expected_test = {
    "workflow_error": 138,
    "constraint_error": 70,
    "tool_use_error": 38,
    "grounding_state_error": 30,
    "reasoning_value_error": 11,
}

assert (
    train_df["failure_family"].value_counts().to_dict()
    == expected_train
)

assert (
    test_df["failure_family"].value_counts().to_dict()
    == expected_test
)

# ---------------------------------------------------------
# 3. Label mapping consistency
# ---------------------------------------------------------

label_map = {
    0: "workflow_error",
    1: "constraint_error",
    2: "tool_use_error",
    3: "grounding_state_error",
    4: "reasoning_value_error",
}

for df in [train_df, test_df]:

    reconstructed = df["family_label"].map(label_map)

    assert (
        reconstructed.values
        == df["failure_family"].values
    ).all()

# ---------------------------------------------------------
# 4. Group leakage
# ---------------------------------------------------------

train_groups = set(train_df["canonical_group"])
test_groups = set(test_df["canonical_group"])

overlap = train_groups & test_groups

print("Train groups:", len(train_groups))
print("Test groups:", len(test_groups))
print("Overlap:", len(overlap))

assert len(overlap) == 0

# ---------------------------------------------------------
# 5. Key uniqueness
# ---------------------------------------------------------

key_cols = [
    "dataset",
    "group_id",
    "message_index",
]

print(
    "Train duplicate keys:",
    train_df.duplicated(key_cols).sum()
)

print(
    "Test duplicate keys:",
    test_df.duplicated(key_cols).sum()
)

assert train_df.duplicated(key_cols).sum() == 0
assert test_df.duplicated(key_cols).sum() == 0

# ---------------------------------------------------------
# 6. Text integrity
# ---------------------------------------------------------

assert train_df["current_text"].notna().all()
assert test_df["current_text"].notna().all()

# Context may legitimately be empty.
print(
    "Empty train contexts:",
    train_df["context_text"].fillna("").eq("").sum()
)

print(
    "Empty test contexts:",
    test_df["context_text"].fillna("").eq("").sum()
)

# ---------------------------------------------------------
# Final
# ---------------------------------------------------------

print("\n✓ DATASET INTEGRITY CHECK PASSED")
print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train groups: 335
Test groups: 84
Overlap: 0
Train duplicate keys: 0
Test duplicate keys: 0
Empty train contexts: 56
Empty test contexts: 18

✓ DATASET INTEGRITY CHECK PASSED
Train: (1489, 42)
Test: (287, 42)


In [6]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
N_CLASSES = 5

family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

y_train = train_df["family_label"].to_numpy()
y_test = test_df["family_label"].to_numpy()

groups_train = train_df["canonical_group"].to_numpy()
groups_test = test_df["canonical_group"].to_numpy()

In [7]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

reason_model = SentenceTransformer(
    MODEL_NAME
)

train_texts = (
    train_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_texts = (
    test_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_train_semantic = reason_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_test_semantic = reason_model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(X_train_semantic.shape)
print(X_test_semantic.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(1489, 384)
(287, 384)


In [8]:
categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",
]

numeric_features = [
    "message_index",
    "is_tool_call",

    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",

    "parsed_tool_calls_in_context",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",

    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",
]

structured_features = (
    categorical_features
    + numeric_features
)

missing = [
    c
    for c in structured_features
    if c not in train_df.columns
]

print("Missing:", missing)

assert not missing

Missing: []


In [9]:
X_train_structured_df = (
    train_df[structured_features]
    .copy()
)

X_test_structured_df = (
    test_df[structured_features]
    .copy()
)

In [10]:
def make_semantic_model():

    return LogisticRegression(
        max_iter=5000,
        class_weight=None,
        random_state=RANDOM_STATE,
    )


def make_structured_model():

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
            ),
        ),
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features,
        ),
    ])

    return Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

In [11]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

semantic_oof = np.zeros(
    (len(train_df), N_CLASSES)
)

structured_oof = np.zeros(
    (len(train_df), N_CLASSES)
)

In [12]:
for fold, (fit_idx, val_idx) in enumerate(
    cv.split(
        X_train_semantic,
        y_train,
        groups_train,
    ),
    start=1,
):

    print(
        f"Fold {fold}:",
        len(fit_idx),
        len(val_idx),
    )

    assert set(
        groups_train[fit_idx]
    ).isdisjoint(
        set(groups_train[val_idx])
    )

    # Semantic
    sem_model = make_semantic_model()

    sem_model.fit(
        X_train_semantic[fit_idx],
        y_train[fit_idx],
    )

    semantic_oof[val_idx] = (
        sem_model.predict_proba(
            X_train_semantic[val_idx]
        )
    )

    # Structured
    struct_model = make_structured_model()

    struct_model.fit(
        X_train_structured_df.iloc[
            fit_idx
        ],
        y_train[fit_idx],
    )

    structured_oof[val_idx] = (
        struct_model.predict_proba(
            X_train_structured_df.iloc[
                val_idx
            ]
        )
    )

Fold 1: 1190 299
Fold 2: 1191 298
Fold 3: 1191 298
Fold 4: 1192 297
Fold 5: 1192 297


In [13]:
assert np.allclose(
    semantic_oof.sum(axis=1),
    1.0,
)

assert np.allclose(
    structured_oof.sum(axis=1),
    1.0,
)

print("✓ OOF predictions valid")

✓ OOF predictions valid


In [14]:
X_meta_train = np.hstack([
    semantic_oof,
    structured_oof,
])

meta_model = LogisticRegression(
    max_iter=5000,
    class_weight=None,
    random_state=RANDOM_STATE,
)

meta_model.fit(
    X_meta_train,
    y_train,
)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [16]:
semantic_model = make_semantic_model()

semantic_model.fit(
    X_train_semantic,
    y_train,
)


structured_model = make_structured_model()

structured_model.fit(
    X_train_structured_df,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](5,)","[0,1,2,3,4]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['current_role','current_tool','previous_tool',...,'same_tool_as_previous', 'current_tool_previous_count','current_action_seen_before']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropp

In [17]:
semantic_test_proba = (
    semantic_model.predict_proba(
        X_test_semantic
    )
)

structured_test_proba = (
    structured_model.predict_proba(
        X_test_structured_df
    )
)

X_meta_test = np.hstack([
    semantic_test_proba,
    structured_test_proba,
])

semantic_pred = np.argmax(
    semantic_test_proba,
    axis=1,
)

structured_pred = np.argmax(
    structured_test_proba,
    axis=1,
)

stacked_pred = meta_model.predict(
    X_meta_test
)

In [18]:
def metrics_dict(
    y_true,
    y_pred,
):

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

In [19]:
model_results = pd.DataFrame([
    {
        "model": "semantic",
        **metrics_dict(
            y_test,
            semantic_pred,
        ),
    },
    {
        "model": "structured",
        **metrics_dict(
            y_test,
            structured_pred,
        ),
    },
    {
        "model": "stacked",
        **metrics_dict(
            y_test,
            stacked_pred,
        ),
    },
])

model_results

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic,0.515679,0.481391,0.490268,0.521487
1,structured,0.439024,0.432212,0.434730,0.450732
2,stacked,0.515679,0.475764,0.492596,0.518409


In [20]:
analysis_df = test_df[
    [
        "dataset",
        "canonical_group",
        "group_id",
        "message_index",
        "failure_family",
        "current_text",
    ]
].copy()

analysis_df["y_true"] = y_test
analysis_df["semantic_pred"] = semantic_pred
analysis_df["structured_pred"] = structured_pred
analysis_df["stacked_pred"] = stacked_pred

analysis_df["semantic_correct"] = (
    semantic_pred == y_test
)

analysis_df["structured_correct"] = (
    structured_pred == y_test
)

analysis_df["stacked_correct"] = (
    stacked_pred == y_test
)

analysis_df["branches_disagree"] = (
    semantic_pred != structured_pred
)

In [21]:
print(
    "Branch disagreement:",
    analysis_df[
        "branches_disagree"
    ].mean()
)

print(
    "Rows:",
    analysis_df[
        "branches_disagree"
    ].sum(),
)

Branch disagreement: 0.40418118466898956
Rows: 116


In [22]:
complementarity = pd.crosstab(
    analysis_df[
        "semantic_correct"
    ],
    analysis_df[
        "structured_correct"
    ],
)

complementarity

structured_correct,False,True
semantic_correct,,
False,110,29
True,51,97


In [23]:
both_correct = (
    analysis_df["semantic_correct"]
    &
    analysis_df["structured_correct"]
)

semantic_only = (
    analysis_df["semantic_correct"]
    &
    ~analysis_df["structured_correct"]
)

structured_only = (
    ~analysis_df["semantic_correct"]
    &
    analysis_df["structured_correct"]
)

both_wrong = (
    ~analysis_df["semantic_correct"]
    &
    ~analysis_df["structured_correct"]
)

complementarity_summary = pd.DataFrame([
    {
        "case": "both_correct",
        "count": both_correct.sum(),
    },
    {
        "case": "semantic_only_correct",
        "count": semantic_only.sum(),
    },
    {
        "case": "structured_only_correct",
        "count": structured_only.sum(),
    },
    {
        "case": "both_wrong",
        "count": both_wrong.sum(),
    },
])

complementarity_summary[
    "percentage"
] = (
    complementarity_summary["count"]
    / len(test_df)
    * 100
)

complementarity_summary

,case,count,percentage
0,both_correct,97,33.797909
1,semantic_only_correct,51,17.770035
2,structured_only_correct,29,10.104530
3,both_wrong,110,38.327526


In [24]:
analysis_df[
    "stack_rescues_semantic"
] = (
    ~analysis_df["semantic_correct"]
    &
    analysis_df["stacked_correct"]
)

analysis_df[
    "stack_breaks_semantic"
] = (
    analysis_df["semantic_correct"]
    &
    ~analysis_df["stacked_correct"]
)

In [25]:
print(
    "Stack rescues semantic errors:",
    analysis_df[
        "stack_rescues_semantic"
    ].sum(),
)

print(
    "Stack breaks semantic successes:",
    analysis_df[
        "stack_breaks_semantic"
    ].sum(),
)

print(
    "Net:",
    analysis_df[
        "stack_rescues_semantic"
    ].sum()
    -
    analysis_df[
        "stack_breaks_semantic"
    ].sum(),
)

Stack rescues semantic errors: 16
Stack breaks semantic successes: 16
Net: 0


In [26]:
rescue_by_class = []

for class_id, class_name in enumerate(
    family_names
):

    mask = (
        y_test == class_id
    )

    rescue_by_class.append({
        "failure_family": class_name,

        "support":
            mask.sum(),

        "semantic_correct":
            analysis_df.loc[
                mask,
                "semantic_correct"
            ].sum(),

        "structured_only_correct":
            structured_only[
                mask
            ].sum(),

        "stack_rescues":
            analysis_df.loc[
                mask,
                "stack_rescues_semantic"
            ].sum(),

        "stack_breaks":
            analysis_df.loc[
                mask,
                "stack_breaks_semantic"
            ].sum(),
    })

rescue_by_class = pd.DataFrame(
    rescue_by_class
)

rescue_by_class

,failure_family,support,semantic_correct,structured_only_correct,stack_rescues,stack_breaks
0,workflow_error,138,76,15,10,7
1,constraint_error,70,43,5,3,6
2,tool_use_error,38,10,4,1,0
3,grounding_state_error,30,13,5,2,3
4,reasoning_value_error,11,6,0,0,0


In [27]:
oracle_correct = (
    analysis_df["semantic_correct"]
    |
    analysis_df["structured_correct"]
)

oracle_accuracy = (
    oracle_correct.mean()
)

print(
    "Semantic accuracy:",
    analysis_df[
        "semantic_correct"
    ].mean()
)

print(
    "Structured accuracy:",
    analysis_df[
        "structured_correct"
    ].mean()
)

print(
    "Stack accuracy:",
    analysis_df[
        "stacked_correct"
    ].mean()
)

print(
    "Oracle branch accuracy:",
    oracle_accuracy
)

Semantic accuracy: 0.5156794425087108
Structured accuracy: 0.43902439024390244
Stack accuracy: 0.5156794425087108
Oracle branch accuracy: 0.6167247386759582


In [28]:
test_groups_unique = (
    test_df["canonical_group"]
    .unique()
)

len(test_groups_unique)

84

In [29]:
rng = np.random.default_rng(
    RANDOM_STATE
)

N_BOOTSTRAP = 5000

bootstrap_rows = []

In [30]:
for b in range(N_BOOTSTRAP):

    sampled_groups = rng.choice(
        test_groups_unique,
        size=len(test_groups_unique),
        replace=True,
    )

    sampled_indices = []

    # Important:
    # duplicated groups must contribute
    # duplicated rows to bootstrap sample.
    for group in sampled_groups:

        idx = np.flatnonzero(
            groups_test == group
        )

        sampled_indices.extend(
            idx.tolist()
        )

    sampled_indices = np.asarray(
        sampled_indices
    )

    y_b = y_test[
        sampled_indices
    ]

    semantic_b = semantic_pred[
        sampled_indices
    ]

    structured_b = structured_pred[
        sampled_indices
    ]

    stacked_b = stacked_pred[
        sampled_indices
    ]

    semantic_f1 = f1_score(
        y_b,
        semantic_b,
        average="macro",
        zero_division=0,
    )

    structured_f1 = f1_score(
        y_b,
        structured_b,
        average="macro",
        zero_division=0,
    )

    stacked_f1 = f1_score(
        y_b,
        stacked_b,
        average="macro",
        zero_division=0,
    )

    bootstrap_rows.append({
        "semantic_macro_f1":
            semantic_f1,

        "structured_macro_f1":
            structured_f1,

        "stacked_macro_f1":
            stacked_f1,

        "stack_minus_semantic":
            stacked_f1
            - semantic_f1,

        "stack_minus_structured":
            stacked_f1
            - structured_f1,
    })

In [31]:
bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

bootstrap_df.describe(
    percentiles=[
        .025,
        .05,
        .5,
        .95,
        .975,
    ]
)

,semantic_macro_f1,structured_macro_f1,stacked_macro_f1,stack_minus_semantic,stack_minus_structured
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.480242,0.425262,0.482415,0.002173,0.057153
std,0.042371,0.036514,0.040359,0.016262,0.024023
min,0.270719,0.250378,0.293938,-0.055424,-0.032418
2.5%,0.392709,0.348667,0.400235,-0.029740,0.011744
5%,0.408969,0.363299,0.414035,-0.024619,0.017769
50%,0.482444,0.426596,0.483817,0.001958,0.057087
95%,0.548748,0.482088,0.545309,0.028717,0.096930
97.5%,0.558794,0.492951,0.555136,0.033790,0.105987
max,0.611911,0.539875,0.608088,0.066253,0.151330


In [32]:
delta = bootstrap_df[
    "stack_minus_semantic"
]

ci_low = delta.quantile(
    0.025
)

ci_high = delta.quantile(
    0.975
)

mean_delta = delta.mean()

print(
    "Mean Δ Macro F1:",
    mean_delta
)

print(
    "95% CI:",
    (
        ci_low,
        ci_high,
    )
)

print(
    "P(stack > semantic):",
    (delta > 0).mean()
)

Mean Δ Macro F1: 0.002172609994601125
95% CI: (-0.02974011876504365, 0.033789680500534845)
P(stack > semantic): 0.5534


In [33]:
accuracy_bootstrap = []

for b in range(N_BOOTSTRAP):

    sampled_groups = rng.choice(
        test_groups_unique,
        size=len(test_groups_unique),
        replace=True,
    )

    sampled_indices = np.concatenate([
        np.flatnonzero(
            groups_test == group
        )
        for group in sampled_groups
    ])

    semantic_acc = accuracy_score(
        y_test[sampled_indices],
        semantic_pred[sampled_indices],
    )

    stacked_acc = accuracy_score(
        y_test[sampled_indices],
        stacked_pred[sampled_indices],
    )

    accuracy_bootstrap.append(
        stacked_acc - semantic_acc
    )

In [34]:
accuracy_bootstrap = np.asarray(
    accuracy_bootstrap
)

print(
    "Accuracy Δ mean:",
    accuracy_bootstrap.mean()
)

print(
    "Accuracy Δ 95% CI:",
    np.quantile(
        accuracy_bootstrap,
        [0.025, 0.975],
    )
)

print(
    "P(stack > semantic):",
    (
        accuracy_bootstrap > 0
    ).mean()
)

Accuracy Δ mean: -0.00010272645677066974
Accuracy Δ 95% CI: [-0.0383278   0.04137931]
P(stack > semantic): 0.4588


In [35]:
rescued = analysis_df[
    analysis_df[
        "stack_rescues_semantic"
    ]
].copy()

rescued[
    [
        "dataset",
        "failure_family",
        "semantic_pred",
        "structured_pred",
        "stacked_pred",
        "current_text",
    ]
].head(30)

,dataset,failure_family,semantic_pred,structured_pred,stacked_pred,current_text
40,A,grounding_state_error,1,3,3,[ASSISTANT]\n<answer>The Joe Schmo Show</answer>
54,A,grounding_state_error,1,3,3,[ASSISTANT]\n<answer>Richmond River</answer>
81,B,constraint_error,0,1,1,"[ASSISTANT]\n{""message"":""Reservation 887654321..."
89,B,constraint_error,0,1,1,[ASSISTANT]\n6
99,B,workflow_error,1,0,0,"[ASSISTANT]\nTo confirm, you would like to pro..."
137,B,constraint_error,0,1,1,"[ASSISTANT]\n{""message"":""Thanks — to confirm, ..."
162,B,tool_use_error,0,2,2,[TOOL_CALL]\n\ncheck_network_mode_preference({})
165,B,workflow_error,2,0,0,[TOOL_CALL]\n\ncheck_wifi_status({})
175,B,workflow_error,2,0,0,[TOOL_CALL]\n\ncheck_wifi_status({})
185,B,workflow_error,2,0,0,[TOOL_CALL]\n\ncheck_wifi_status({})


In [36]:
for col in [
    "semantic_pred",
    "structured_pred",
    "stacked_pred",
]:

    rescued[
        f"{col}_name"
    ] = rescued[col].map(
        label_map
    )

In [37]:
rescued[
    [
        "dataset",
        "failure_family",
        "semantic_pred_name",
        "structured_pred_name",
        "stacked_pred_name",
        "current_text",
    ]
].head(30)

,dataset,failure_family,semantic_pred_name,structured_pred_name,stacked_pred_name,current_text
40,A,grounding_state_error,constraint_error,grounding_state_error,grounding_state_error,[ASSISTANT]\n<answer>The Joe Schmo Show</answer>
54,A,grounding_state_error,constraint_error,grounding_state_error,grounding_state_error,[ASSISTANT]\n<answer>Richmond River</answer>
81,B,constraint_error,workflow_error,constraint_error,constraint_error,"[ASSISTANT]\n{""message"":""Reservation 887654321..."
89,B,constraint_error,workflow_error,constraint_error,constraint_error,[ASSISTANT]\n6
99,B,workflow_error,constraint_error,workflow_error,workflow_error,"[ASSISTANT]\nTo confirm, you would like to pro..."
137,B,constraint_error,workflow_error,constraint_error,constraint_error,"[ASSISTANT]\n{""message"":""Thanks — to confirm, ..."
162,B,tool_use_error,workflow_error,tool_use_error,tool_use_error,[TOOL_CALL]\n\ncheck_network_mode_preference({})
165,B,workflow_error,tool_use_error,workflow_error,workflow_error,[TOOL_CALL]\n\ncheck_wifi_status({})
175,B,workflow_error,tool_use_error,workflow_error,workflow_error,[TOOL_CALL]\n\ncheck_wifi_status({})
185,B,workflow_error,tool_use_error,workflow_error,workflow_error,[TOOL_CALL]\n\ncheck_wifi_status({})


In [38]:
broken = analysis_df[
    analysis_df[
        "stack_breaks_semantic"
    ]
].copy()

broken[
    [
        "dataset",
        "failure_family",
        "current_text",
    ]
].head(30)

,dataset,failure_family,current_text
1,A,grounding_state_error,[ASSISTANT]\n<answer>Firth of Forth</answer>
38,A,workflow_error,[TOOL_CALL]\n<thinking>\nI need to reason abou...
60,B,workflow_error,"[TOOL_CALL]\n\nsearch_onestop_flight({""origin""..."
65,B,constraint_error,"[TOOL_CALL]\n\nbook_reservation({""user_id"": ""m..."
68,B,constraint_error,"[TOOL_CALL]\n\nbook_reservation({""user_id"": ""m..."
74,B,workflow_error,"[ASSISTANT]\nI'm sorry, but I couldn't find th..."
79,B,workflow_error,"[ASSISTANT]\n{""message"":""I will (1) calculate ..."
112,B,constraint_error,[ASSISTANT]\nYour flight has been successfully...
118,B,constraint_error,"[TOOL_CALL]\n\ntransfer_to_human_agents({""summ..."
148,B,constraint_error,"[TOOL_CALL]\n\ntransfer_to_human_agents({""summ..."


In [39]:
from pathlib import Path

RESULTS_DIR = Path(
    "../results/experiment_10"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [40]:
model_results.to_csv(
    RESULTS_DIR
    / "model_results.csv",
    index=False,
)

analysis_df.to_csv(
    RESULTS_DIR
    / "test_complementarity.csv",
    index=False,
)

complementarity_summary.to_csv(
    RESULTS_DIR
    / "complementarity_summary.csv",
    index=False,
)

rescue_by_class.to_csv(
    RESULTS_DIR
    / "rescue_by_class.csv",
    index=False,
)

bootstrap_df.to_csv(
    RESULTS_DIR
    / "bootstrap_macro_f1.csv",
    index=False,
)

In [41]:
summary = pd.DataFrame([
    {
        "comparison":
            "stacked_minus_semantic",

        "observed_delta_macro_f1":
            f1_score(
                y_test,
                stacked_pred,
                average="macro",
            )
            -
            f1_score(
                y_test,
                semantic_pred,
                average="macro",
            ),

        "bootstrap_mean_delta":
            mean_delta,

        "ci_2.5":
            ci_low,

        "ci_97.5":
            ci_high,

        "probability_delta_positive":
            (delta > 0).mean(),
    }
])

summary

,comparison,observed_delta_macro_f1,bootstrap_mean_delta,ci_2.5,ci_97.5,probability_delta_positive
0,stacked_minus_semantic,0.002328,0.002173,-0.02974,0.03379,0.5534


In [42]:
summary.to_csv(
    RESULTS_DIR
    / "stability_summary.csv",
    index=False,
)

## Experiment 10 — Fusion Diagnostics and Statistical Validation

### Objective

The goal of Experiment 10 was to determine whether combining **semantic information from the current message** with **structured relational/trajectory information** provides a reliable improvement for failure-family classification.

Previous experiments showed that:

* semantic current-message representations were the strongest standalone baseline;
* structured trajectory features contained useful but weaker predictive information;
* early fusion produced only modest improvements;
* group-safe OOF stacking suggested that semantic and structured representations might contain complementary information.

Experiment 10 therefore focused on understanding whether this complementarity is **real, statistically meaningful, and exploitable by fusion**.

---

### Dataset and Evaluation Protocol

The experiment used the canonical group-safe dataset split:

* **1,776 total examples**
* **1,489 training examples**
* **287 test examples**
* **335 training groups**
* **84 test groups**
* **0 group overlap**

The five failure families were:

1. `workflow_error`
2. `constraint_error`
3. `tool_use_error`
4. `grounding_state_error`
5. `reasoning_value_error`

Dataset integrity checks confirmed that labels, keys, text fields, and group assignments were consistent and that there was no train/test trajectory leakage.

---

### Models Compared

Three prediction systems were evaluated:

* **Semantic model** — classification using the semantic representation of the current message.
* **Structured model** — classification using structured relational and trajectory features.
* **Stacked model** — late fusion combining semantic and structured model predictions using group-safe out-of-fold predictions.

The test-set results were:

| Model      |   Accuracy | Balanced Accuracy |   Macro F1 | Weighted F1 |
| ---------- | ---------: | ----------------: | ---------: | ----------: |
| Semantic   | **0.5157** |        **0.4814** |     0.4903 |  **0.5215** |
| Structured |     0.4390 |            0.4322 |     0.4347 |      0.4507 |
| Stacked    | **0.5157** |            0.4758 | **0.4926** |      0.5184 |

The stacked model achieved only a very small Macro-F1 improvement over the semantic baseline:

[
\Delta MacroF1 \approx +0.0023
]

while accuracy remained unchanged.

---

### Branch Complementarity

Despite the weak aggregate improvement from stacking, the two branches frequently disagreed:

[
40.4%
]

of test examples received different predictions from the semantic and structured classifiers.

Their correctness breakdown was:

| Case                    | Count | Percentage |
| ----------------------- | ----: | ---------: |
| Both correct            |    97 |      33.8% |
| Semantic only correct   |    51 |      17.8% |
| Structured only correct |    29 |      10.1% |
| Both wrong              |   110 |      38.3% |

This is an important result.

The structured representation correctly classified **29 examples that the stronger semantic classifier missed**. Therefore, structured trajectory information is not simply a weaker duplicate of semantic information; it captures genuinely complementary signals.

---

### Oracle Analysis

An oracle classifier was constructed conceptually by counting an example as correct whenever **either branch** predicted the correct label.

The resulting accuracies were:

* Semantic: **51.57%**
* Structured: **43.90%**
* Stacked: **51.57%**
* Oracle branch selector: **61.67%**

Thus, perfect branch selection would theoretically provide:

[
61.67 - 51.57 = \mathbf{10.10}
]

percentage points of accuracy improvement over the semantic baseline.

This demonstrates that the primary fusion problem is not a lack of complementary information.

Instead, the challenge is learning **when the structured branch should override the semantic branch**.

---

### Rescue and Regression Analysis

The stacked classifier rescued **16 semantic-model errors**, but it also broke **16 examples that the semantic model originally classified correctly**.

Therefore:

[
16\text{ rescues} - 16\text{ regressions} = 0
]

net accuracy improvement.

This explains why stacking failed to improve overall accuracy despite the complementary information available in the structured representation.

The structured branch provided unique correct predictions particularly for:

* `workflow_error`: 15 examples
* `constraint_error`: 5 examples
* `tool_use_error`: 4 examples
* `grounding_state_error`: 5 examples
* `reasoning_value_error`: 0 examples

This suggests that trajectory information is especially relevant for errors involving **workflow progression, tool usage, and state/grounding relationships**.

---

### Bootstrap Statistical Analysis

Bootstrap resampling was used to estimate uncertainty around the difference between stacked and semantic performance.

For Macro F1:

[
\text{Observed }\Delta = +0.0023
]

[
\text{Bootstrap mean }\Delta = +0.00217
]

[
95%,CI = [-0.0297,\ 0.0338]
]

and:

[
P(\text{stacked} > \text{semantic}) = 0.5534
]

For accuracy:

[
\text{mean }\Delta \approx -0.0001
]

with:

[
95%,CI = [-0.0383,\ 0.0414]
]

The confidence intervals contain zero by a substantial margin. Therefore, the experiment provides **no statistical evidence that the stacked classifier reliably outperforms the semantic baseline**.

The small observed Macro-F1 improvement should consequently not be interpreted as a meaningful model improvement.

---

### Key Finding

The most important result of Experiment 10 is the distinction between **representation complementarity** and **successful fusion**.

The experiment demonstrates that:

> Semantic and structured trajectory representations contain complementary information, but the current linear stacking strategy cannot reliably determine when each representation should be trusted.

The 61.67% oracle accuracy shows that substantial predictive information remains available across the two branches. However, simple stacking captures almost none of this theoretical gain.

At the same time, the 110 examples (**38.3% of the test set**) that both branches classify incorrectly indicate a second limitation: many failures likely require richer representations of tool results, state transitions, constraints, arguments, or relationships between previous evidence and the current action.

---

## Conclusion

Experiment 10 does **not** support the claim that stacked fusion improves classification performance over the semantic baseline. The observed Macro-F1 difference is small and statistically uncertain.

However, the experiment provides a more important research result: the semantic and structured classifiers make meaningfully different errors. A perfect selector between them would increase accuracy from **51.57% to 61.67%**, revealing a substantial **10.1 percentage-point oracle gap**.

Therefore, the next research problem is no longer simply:

> *Can semantic and structured features be combined?*

Instead, it becomes:

> **Can the model learn when semantic evidence is sufficient and when trajectory-level structural evidence should dominate the prediction?**

This motivates the next experiment using **gated fusion / mixture-of-experts**, where a learned gating mechanism explicitly estimates which representation should receive greater weight for each example.
